# Planetary-dominated eddies: vertical background flow and tilt

This notebook asks whether depth-dependent environmental flow can explain the shared zonal component of planetary-dominated eddy tilt, while the beta effect explains the opposite meridional components. It uses exactly the same 2:1 planetary-dominance, 3000 m minimum depth, and 5 km minimum tilt criteria as `plan_topo_dom_eddies/planetary_dominated_eddies.ipynb`.

`TiltDir` is a compass bearing **from the deep centre to the shallow centre** (0° north, 90° east). Therefore the kinematic predictor is shallow flow minus deep flow. A deeper layer flowing farther west than the surface gives a positive/eastward predictor, matching an eastward shallow-relative-to-deep tilt.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
FLOW_ROOT = ANALYSIS_ROOT / 'beta_effect_background_flow'
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (ANALYSIS_ROOT, FLOW_ROOT, CASE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from analysis_tools import add_track_velocity
from background_flow_tools import BackgroundConfig, load_background_cache
from case_study_tools import PVAlignmentConfig, add_pv_alignment_diagnostics

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 100)

## 1. Settings and exact planetary-dominated sample

In [ ]:
DOMINANCE_FACTOR = 2.0
MIN_DEPTH_m = 3000.0
MIN_TILT_km = 5.0
PRIMARY_FAMILY = 'ann'       # instantaneous eddy-following annulus
FAMILIES = ('ann', 'clim', 'full')
N_BOOT = 5000
RANDOM_SEED = 42

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
eddies = tilt.add_region_labels(eddies, grid)
eddies = tilt.add_pv_gradient_terms(eddies, grid, core_mean=True)

# Calculate track derivatives before filtering so boundary days are not created artificially.
eddies = add_track_velocity(eddies, grid.angle, window=5)
background = load_background_cache(BackgroundConfig())
required = {f'{family}_{depth}_{component}_ms' for family in FAMILIES
            for depth in ('surface', '200', '500') for component in ('east', 'north')}
missing = required - set(background.columns)
if missing:
    raise RuntimeError(f'Background cache is missing {sorted(missing)}. Run 01_build_background_cache.ipynb first.')
d = eddies.merge(background, on=['Eddy', 'Day'], how='inner', validate='one_to_one')
pv_config = PVAlignmentConfig(dominance_factor=DOMINANCE_FACTOR,
                              min_open_ocean_depth_m=MIN_DEPTH_m,
                              min_tilt_distance_km=MIN_TILT_km)
d = add_pv_alignment_diagnostics(d, pv_config)
planetary = d[d.open_ocean_planetary & d.direction_valid].copy()
theta = np.deg2rad(planetary.TiltDir.astype(float))
planetary['tilt_east_km'] = planetary.TiltDis * np.sin(theta)
planetary['tilt_north_km'] = planetary.TiltDis * np.cos(theta)
display(planetary.groupby('Cyc').agg(observations=('Eddy','size'), eddies=('Eddy','nunique'),
    median_depth_m=('h','median'), median_tilt_east_km=('tilt_east_km','median'),
    median_tilt_north_km=('tilt_north_km','median')).round(2))

## 2. Construct physically interpretable layers and shear

The cache contains a surface value and thickness-weighted 0–200 m and 0–500 m means. Because every selected water column is deeper than 3000 m, the 200–500 m mean can be recovered as `(500 × mean_0_500 − 200 × mean_0_200) / 300`. This is a **layer mean**, not a velocity at a single sigma or z level.

The principal predictor is `surface − 200–500 m`, because the measured tilt points from deep to shallow. Positive east shear predicts eastward tilt; positive north shear predicts northward tilt. The three background families are sensitivity tests: instantaneous annulus (`ann`), calendar-month climatology (`clim`), and full 26-year mean (`full`).

In [ ]:
for family in FAMILIES:
    for component in ('east', 'north'):
        surface = f'{family}_surface_{component}_ms'
        upper = f'{family}_200_{component}_ms'
        full500 = f'{family}_500_{component}_ms'
        deep = f'{family}_200_500_{component}_ms'
        planetary[deep] = (500 * planetary[full500] - 200 * planetary[upper]) / 300
        planetary[f'{family}_surface_minus_200_{component}_ms'] = planetary[surface] - planetary[upper]
        planetary[f'{family}_surface_minus_deep_{component}_ms'] = planetary[surface] - planetary[deep]
        planetary[f'{family}_upper_minus_deep_{component}_ms'] = planetary[upper] - planetary[deep]
        # These are track velocities relative to different environmental references,
        # not independent measurements of the shallow and deep eddy-centre motion.
        planetary[f'{family}_surface_residual_{component}_ms'] = planetary[f'track_{component}_ms'] - planetary[surface]
        planetary[f'{family}_deep_residual_{component}_ms'] = planetary[f'track_{component}_ms'] - planetary[deep]

depth_labels = {'surface': 'Surface', '200': '0–200 m', '200_500': '200–500 m'}
planetary.filter(regex='^(Eddy|Day|Cyc|Tilt|tilt_|ann_.*(east|north)_ms)$').head()

## 3. Flow at each depth

The bars use one median per eddy and then summarize eddies, avoiding long-lived eddies dominating the result. Compare AE and CE at each depth. Similar eastward profiles support a shared zonal mechanism; opposite northward profiles would support polarity-dependent sampling or dynamics.

In [ ]:
def eddy_medians(frame, columns):
    return frame.groupby(['Cyc', 'Eddy'], as_index=False)[columns].median(numeric_only=True)

def bootstrap_median(values, n_boot=N_BOOT, seed=RANDOM_SEED):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.median(rng.choice(values, (n_boot, len(values)), replace=True), axis=1)
    return np.median(values), *np.percentile(boot, [2.5, 97.5])

def depth_summary(frame, family):
    rows = []
    for component in ('east', 'north'):
        for depth in depth_labels:
            col = f'{family}_{depth}_{component}_ms'
            em = eddy_medians(frame, [col])
            for cyc, part in em.groupby('Cyc'):
                med, lo, hi = bootstrap_median(part[col], seed=RANDOM_SEED + len(rows))
                rows.append(dict(family=family, component=component, depth=depth, Cyc=cyc,
                                 eddies=part[col].notna().sum(), median=med, ci_low=lo, ci_high=hi))
    return pd.DataFrame(rows)

flow_summary = pd.concat([depth_summary(planetary, f) for f in FAMILIES], ignore_index=True)
display(flow_summary.round(4))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey='row', constrained_layout=True)
colors = {'AE': 'firebrick', 'CE': 'royalblue'}
for j, family in enumerate(FAMILIES):
    for i, component in enumerate(('east', 'north')):
        ax = axes[i, j]
        part = flow_summary.query('family == @family and component == @component')
        for offset, cyc in zip((-0.08, 0.08), ('AE', 'CE')):
            q = part[part.Cyc.eq(cyc)].set_index('depth').reindex(depth_labels)
            x = np.arange(3) + offset
            ax.errorbar(x, q['median'], yerr=[q['median']-q['ci_low'], q['ci_high']-q['median']],
                        fmt='o-', capsize=3, color=colors[cyc], label=cyc)
        ax.axhline(0, color='k', lw=.8)
        ax.set_xticks(range(3), depth_labels.values(), rotation=15)
        ax.set_title(f'{family}: {component}ward flow')
        if j == 0: ax.set_ylabel('Velocity (m s$^{-1}$)')
        if i == 0 and j == 0: ax.legend()
plt.show()

## 4. Does surface-minus-deep shear have the observed tilt direction?

The first panel compares the AE/CE median shear vectors with the observed tilt vectors. Arrow directions, rather than lengths, are directly comparable because velocity and displacement have different units. The second and third panels test zonal and meridional components at the eddy level.

In [ ]:
family = PRIMARY_FAMILY
shear_e = f'{family}_surface_minus_deep_east_ms'
shear_n = f'{family}_surface_minus_deep_north_ms'
cols = ['tilt_east_km', 'tilt_north_km', shear_e, shear_n]
eddy_level = eddy_medians(planetary, cols)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
ax = axes[0]
for cyc in ('AE', 'CE'):
    q = eddy_level[eddy_level.Cyc.eq(cyc)]
    se, sn = q[[shear_e, shear_n]].median()
    te, tn = q[['tilt_east_km', 'tilt_north_km']].median()
    # Normalize each arrow: this panel compares direction only.
    for e, n, ls, label in ((se, sn, '-', f'{cyc} shear'), (te, tn, '--', f'{cyc} tilt')):
        norm = np.hypot(e, n)
        ax.annotate('', xy=(e/norm, n/norm), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=colors[cyc], lw=2, linestyle=ls))
        ax.text(e/norm*1.08, n/norm*1.08, label, color=colors[cyc], ha='center')
ax.axhline(0, color='k', lw=.7); ax.axvline(0, color='k', lw=.7)
ax.set(xlim=(-1.25,1.25), ylim=(-1.25,1.25), aspect='equal', xlabel='East', ylabel='North',
       title=f'{family}: median vector directions')

for ax, xcol, ycol, label in ((axes[1], shear_e, 'tilt_east_km', 'zonal'),
                                  (axes[2], shear_n, 'tilt_north_km', 'meridional')):
    for cyc in ('AE', 'CE'):
        q = eddy_level[eddy_level.Cyc.eq(cyc)]
        ax.scatter(q[xcol], q[ycol], s=20, alpha=.45, color=colors[cyc], label=cyc)
        valid = q[[xcol, ycol]].dropna()
        rho, p = spearmanr(valid[xcol], valid[ycol]) if len(valid) > 2 else (np.nan, np.nan)
        ax.text(.03, .95-(.09 if cyc == 'CE' else 0), f'{cyc}: ρ={rho:.2f}, p={p:.3g}, n={len(valid)}',
                transform=ax.transAxes, va='top', color=colors[cyc])
    ax.axhline(0, color='k', lw=.7); ax.axvline(0, color='k', lw=.7)
    ax.set_xlabel('Surface − 200–500 m flow (m s$^{-1}$)')
    ax.set_ylabel(f'{label.capitalize()} tilt (km)')
    ax.set_title(f'{label.capitalize()} shear versus tilt')
axes[1].legend()
plt.show()

## 5. Whole-eddy bootstrap tests

A positive zonal shear median means that the surface environment is eastward relative to 200–500 m and therefore predicts eastward tilt. For the meridional component, the beta-effect prediction is made polarity-aware: northward for AEs and southward for CEs. Confidence intervals resample entire eddies. The correlation test also resamples eddies and is more defensible than treating eddy-days as independent.

In [ ]:
def bootstrap_correlation(frame, x, y, n_boot=N_BOOT, seed=RANDOM_SEED):
    q = frame[['Eddy', x, y]].dropna()
    ids = q.Eddy.unique()
    if len(ids) < 3:
        return np.nan, np.nan, np.nan
    observed = spearmanr(q[x], q[y]).statistic
    rng = np.random.default_rng(seed)
    boot = []
    for _ in range(n_boot):
        sampled = rng.choice(ids, len(ids), replace=True)
        b = pd.concat([q[q.Eddy.eq(e)].assign(Eddy=i) for i, e in enumerate(sampled)], ignore_index=True)
        boot.append(spearmanr(b[x], b[y]).statistic)
    return observed, *np.nanpercentile(boot, [2.5, 97.5])

records = []
for family in FAMILIES:
    for cyc in ('AE', 'CE'):
        q = planetary[planetary.Cyc.eq(cyc)].copy()
        q['beta_aligned_shear_ms'] = np.where(cyc == 'AE', 1, -1) * q[f'{family}_surface_minus_deep_north_ms']
        q['beta_aligned_tilt_km'] = np.where(cyc == 'AE', 1, -1) * q['tilt_north_km']
        em = eddy_medians(q, [f'{family}_surface_minus_deep_east_ms', 'tilt_east_km',
                                'beta_aligned_shear_ms', 'beta_aligned_tilt_km'])
        for mechanism, x, y in (
            ('zonal', f'{family}_surface_minus_deep_east_ms', 'tilt_east_km'),
            ('beta-aligned meridional', 'beta_aligned_shear_ms', 'beta_aligned_tilt_km')):
            med, lo, hi = bootstrap_median(em[x], seed=RANDOM_SEED + len(records))
            rho, rho_lo, rho_hi = bootstrap_correlation(em, x, y, seed=RANDOM_SEED + len(records))
            records.append(dict(family=family, Cyc=cyc, mechanism=mechanism, eddies=len(em),
                median_predictor=med, predictor_ci_low=lo, predictor_ci_high=hi,
                spearman_rho=rho, rho_ci_low=rho_lo, rho_ci_high=rho_hi))
tests = pd.DataFrame(records)
display(tests.round(4))

## 6. Natural/background-relative westward propagation

Negative eastward residual velocity means the tracked eddy propagates westward relative to that background estimate. This checks whether the planetary eddies retain a westward propagation anomaly after removing the flow. However, there is only one tracked horizontal centre velocity per eddy-day: subtracting surface and deep backgrounds does **not** independently observe top and bottom eddy propagation. Consequently, the difference between these residuals is algebraically the negative of the background shear and should not be treated as a second independent shear test.

In [ ]:
residual_cols = []
for family in FAMILIES:
    residual_cols += [f'{family}_surface_residual_east_ms', f'{family}_deep_residual_east_ms',
                      f'{family}_surface_residual_north_ms', f'{family}_deep_residual_north_ms']
residual_eddy = eddy_medians(planetary, residual_cols)
rows = []
for family in FAMILIES:
    for component in ('east', 'north'):
        for reference in ('surface', 'deep'):
            col = f'{family}_{reference}_residual_{component}_ms'
            for cyc, part in residual_eddy.groupby('Cyc'):
                med, lo, hi = bootstrap_median(part[col], seed=RANDOM_SEED + len(rows))
                rows.append(dict(family=family, component=component, reference=reference, Cyc=cyc,
                                 median=med, ci_low=lo, ci_high=hi, eddies=part[col].notna().sum()))
residual_summary = pd.DataFrame(rows)
display(residual_summary.round(4))

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey='row', constrained_layout=True)
for j, family in enumerate(FAMILIES):
    for i, component in enumerate(('east', 'north')):
        ax = axes[i, j]
        p = residual_summary.query('family == @family and component == @component')
        for k, cyc in enumerate(('AE', 'CE')):
            q = p[p.Cyc.eq(cyc)].set_index('reference').reindex(['surface','deep'])
            x = np.arange(2) + (-.07 if cyc == 'AE' else .07)
            ax.errorbar(x, q['median'], yerr=[q['median']-q['ci_low'], q['ci_high']-q['median']],
                        fmt='o-', capsize=3, color=colors[cyc], label=cyc)
        ax.axhline(0, color='k', lw=.8)
        ax.set_xticks([0,1], ['surface ref.', '200–500 m ref.'])
        ax.set_title(f'{family}: residual {component}ward')
        if j == 0: ax.set_ylabel('Track − background (m s$^{-1}$)')
        if i == 0 and j == 0: ax.legend()
plt.show()

## 7. Spatial-sampling check

AE/CE flow differences can arise simply because the two populations occupy different locations. These maps show the primary shear at each eddy-day. The latitude-band table then compares one median per eddy within broad spatial bands; interpret bands with few eddies cautiously.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True, constrained_layout=True)
for i, cyc in enumerate(('AE', 'CE')):
    q = planetary[planetary.Cyc.eq(cyc)]
    for j, (col, title) in enumerate(((shear_e, 'east'), (shear_n, 'north'))):
        ax = axes[i, j]
        lim = np.nanpercentile(np.abs(planetary[col]), 98)
        sc = ax.scatter(q.xc, q.yc, c=q[col], s=9, alpha=.6, cmap='RdBu_r', vmin=-lim, vmax=lim)
        ax.contour(grid.X_grid, grid.Y_grid, grid.h, levels=[MIN_DEPTH_m], colors='k', linewidths=.7)
        tilt.lat_lon_contours(ax, grid)
        ax.set_title(f'{cyc}: surface − deep {title} flow')
        ax.set_aspect('equal', adjustable='box')
        fig.colorbar(sc, ax=ax, label='m s$^{-1}$')
plt.show()

spatial = planetary.copy()
spatial['y_band'] = pd.qcut(spatial.yc, 4, duplicates='drop')
spatial_eddy = spatial.groupby(['Cyc','y_band','Eddy'], observed=True, as_index=False)[
    [shear_e, shear_n, 'tilt_east_km', 'tilt_north_km']].median(numeric_only=True)
display(spatial_eddy.groupby(['Cyc','y_band'], observed=True).agg(
    eddies=('Eddy','nunique'), shear_east_ms=(shear_e,'median'), shear_north_ms=(shear_n,'median'),
    tilt_east_km=('tilt_east_km','median'), tilt_north_km=('tilt_north_km','median')).round(3))

## 8. Interpretation guide

Evidence consistent with the proposed zonal mechanism would require: (1) surface-minus-deep eastward flow is positive for both AE and CE, with whole-eddy confidence intervals excluding zero; (2) this result is robust across annulus, monthly climatology, and full-archive backgrounds; (3) eddies with stronger eastward shear tend to have larger eastward tilt; and (4) the relationship is not explained by AE/CE location differences.

For the meridional beta-effect interpretation, AE predictors and tilts should be preferentially north/equatorward while CE predictors and tilts should be south/poleward. A directional median without a shear–tilt association is suggestive of a shared environment, but not evidence that shear causes tilt.

The strongest limitation is that the cache describes environmental flow around the eddy, not the propagation of separately tracked shallow and deep centres. A direct test of ‘the bottom propagates faster westward than the top’ would require retaining the daily centre coordinates at multiple depth levels and differentiating those depth-specific tracks. This notebook tests the plausible differential-advection mechanism with the data currently cached; it does not equate background shear with depth-specific eddy propagation.